# Notebook Overview — Generate CLIP Text Representations

## Purpose

Generate CLIP-based text representations for NExT-QA development-set questions and multiple-choice answer options. These text representations provide a shared semantic embedding space for downstream comparison with video representations generated by the autoencoder and CLIP video pipelines.

## Inputs

* Shared project configuration and constants.
* NExT-QA annotation files (questions, answer choices, and ground-truth labels).
* Development evaluation subset configuration.
* Pretrained CLIP text encoder model from the Hugging Face Transformers library.

## Outputs

* CLIP text representation dataset containing 512-dimensional embeddings for questions and answer choices.
* CLIP text representation summary report.
* Validation results confirming representation dataset integrity.
* Sample representation records for qualitative verification.

## Processing Workflow

1. Initialize the notebook environment and load shared project configuration.
2. Configure the CLIP text representation experiment.
3. Verify the runtime environment and required software dependencies.
4. Prepare the development evaluation subset and construct normalized text input records.
5. Load the pretrained CLIP text encoder model.
6. Generate normalized CLIP text representations for all questions and answer choices.
7. Validate the generated representation dataset.
8. Save the representation dataset to the experiment-specific Google Drive directory.
9. Generate a summary report describing the completed representation dataset.
10. Display representative text representation records for verification.
11. Summarize notebook outputs and generated artifacts.

## Notes

This notebook generates text representations only. Video representations are produced separately by the autoencoder and CLIP video representation notebooks. The generated text representations are consumed by downstream representation-based VideoQA experiments, where they are combined with video representations to evaluate multiple-choice question answering performance.

# Notebook Overview — Generate CLIP Text Representations

## Purpose

TBD.

## Inputs

TBD.

## Outputs

TBD.

## Processing Workflow

TBD.

## Notes

TBD.

### 🔷 Step 1 — Initialize Environment for CLIP Text Representation Generation

* Mount Google Drive and prepare the Colab execution environment.
* Clone the private project repository and move into the repository directory.
* Load shared project configuration constants and utility modules.
* Verify required project paths and local output directories.
* Load NExT-QA annotation metadata used for CLIP text representation generation.
* Combine annotation split files into a unified dataset for downstream processing.
* Display dataset split summary information when verbose output is enabled.
* Prepare the notebook environment for CLIP-based text representation generation.

In [ ]:
# ============================================================
# Step 1: Initialize Environment for CLIP Text Representation Generation
# ============================================================

VERBOSE = True
REQUIRE_L4_GPU = True

import os
import time
from pathlib import Path

import pandas as pd

from google.colab import drive, userdata

print("Initializing notebook environment...")
print("-" * 60)

# ------------------------------------------------------------
# DRIVE MOUNT
# ------------------------------------------------------------

GOOGLE_DRIVE_MOUNT = "/content/drive"

if not os.path.exists(GOOGLE_DRIVE_MOUNT):
    print("Mounting Google Drive...")
    drive.mount(GOOGLE_DRIVE_MOUNT)
else:
    print("Google Drive already mounted.")

# ------------------------------------------------------------
# CLONE REPOSITORY
# ------------------------------------------------------------

REPO_NAME = "videoqa-representation-comparison"
REPO_OWNER = "pgailinas"
REPO_BASE_DIR = "/content"
REPO_DIR = os.path.join(REPO_BASE_DIR, REPO_NAME)

github_token = userdata.get("GITHUB_TOKEN")

if github_token is None:
    raise ValueError("GITHUB_TOKEN not found in Colab Secrets.")

repo_url = (
    f"https://x-access-token:{github_token}"
    f"@github.com/{REPO_OWNER}/{REPO_NAME}.git"
)

os.chdir(REPO_BASE_DIR)

if not os.path.exists(REPO_DIR):

    print("Cloning project repository...")

    !git clone --quiet --filter=blob:none --no-checkout {repo_url}

    os.chdir(REPO_DIR)

    !git sparse-checkout init --cone
    !git sparse-checkout set src datasets outputs
    !git checkout --quiet main

else:

    print("Project repository already available.")
    os.chdir(REPO_DIR)

print(f"Repository ready: {REPO_DIR}")

# ------------------------------------------------------------
# LOAD PROJECT MODULES
# ------------------------------------------------------------

print("\nLoading project configuration and utility modules...")

from src.videoqa_representation_config import *
from src.nextqa_metadata import *

# ------------------------------------------------------------
# OUTPUT SETUP
# ------------------------------------------------------------

OUTPUTS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

required_paths = [
    Path("src"),
    Path("datasets"),
    OUTPUTS_DIR,
    QUESTIONS_DIR,
]

missing_paths = [
    path for path in required_paths
    if not path.exists()
]

if missing_paths:
    for path in missing_paths:
        print(f"Missing required path: {path}")

    raise FileNotFoundError(
        "One or more required project paths are missing."
    )

print("Configuration loaded.")
print("Project paths initialized.")

# ------------------------------------------------------------
# LOAD NExT-QA ANNOTATIONS
# ------------------------------------------------------------

print("\nLoading NExT-QA annotations...")

split_annotations = load_nextqa_split_annotations(
    annotations_dir=QUESTIONS_DIR,
    verbose=VERBOSE,
)

annotations_df = combine_nextqa_annotations(
    split_dataframes=split_annotations,
    verbose=VERBOSE,
)

split_summary_df = summarize_nextqa_splits(
    annotations=annotations_df,
)

print("\nDataset metadata loaded.")
print(f"Annotation records: {len(annotations_df):,}")

if VERBOSE:
    print("\nSplit Summary")
    print("-" * 60)
    display(split_summary_df)

print("\nEnvironment initialization complete.")
print("-" * 60)
print("Notebook ready for CLIP text representation generation.")



### 🔷 Step 2 — Define CLIP Text Representation Configuration

* Load shared constants from the central project configuration file.
* Configure development-subset settings for CLIP-based text representation generation.
* Define the NExT-QA evaluation split used for development experiments.
* Set the development subset size and random seed to ensure reproducible text embedding generation.
* Configure the CLIP text encoder model and supported text input types.
* Define the question and answer-choice fields used for representation generation.
* Configure experiment output locations for CLIP text representation artifacts.
* Display the active CLIP text representation configuration prior to model loading.

In [ ]:
# ============================================================
# Step 2: Define CLIP Text Representation Configuration
# ============================================================

print("Defining CLIP text representation configuration...")

# ------------------------------------------------------------
# CLIP text model configuration
# ------------------------------------------------------------

CLIP_TEXT_MODEL_NAME = "openai/clip-vit-base-patch32"

TEXT_REPRESENTATION_SCOPE = "questions_and_answer_choices"

TEXT_INPUT_TYPES = [
    "question",
    "answer_choice",
]

QUESTION_TEXT_FIELD = QUESTION_COLUMN
ANSWER_CHOICE_COLUMNS = CHOICE_COLUMNS

# ------------------------------------------------------------
# Development configuration
# ------------------------------------------------------------

evaluation_split = EVALUATION_SPLIT
development_subset_size = DEVELOPMENT_SUBSET_SIZE
random_seed = RANDOM_SEED

# ------------------------------------------------------------
# Output configuration
# ------------------------------------------------------------

CLIP_TEXT_EXPERIMENT_DIR = (
    EXPERIMENTS_DRIVE_DIR
    / EXPERIMENT_NAME
    / "clip"
    / "text"
)

CLIP_TEXT_EXPERIMENT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CLIP_TEXT_REPRESENTATIONS_EXPERIMENT_CSV = (
    CLIP_TEXT_EXPERIMENT_DIR
    / "clip_text_representations.csv"
)

CLIP_TEXT_SUMMARY_EXPERIMENT_CSV = (
    CLIP_TEXT_EXPERIMENT_DIR
    / "clip_text_representation_summary.csv"
)

# ------------------------------------------------------------
# Display active configuration
# ------------------------------------------------------------

clip_text_config_summary = {
    "experiment_name": EXPERIMENT_NAME,
    "clip_text_model": CLIP_TEXT_MODEL_NAME,
    "representation_scope": TEXT_REPRESENTATION_SCOPE,
    "text_input_types": ", ".join(TEXT_INPUT_TYPES),
    "evaluation_split": evaluation_split,
    "development_subset_size": development_subset_size,
    "random_seed": random_seed,
    "question_text_field": QUESTION_TEXT_FIELD,
    "answer_choice_columns": ", ".join(ANSWER_CHOICE_COLUMNS),
    "output_directory": str(CLIP_TEXT_EXPERIMENT_DIR),
}

clip_text_config_df = pd.DataFrame(
    clip_text_config_summary.items(),
    columns=["Configuration Item", "Value"],
)

print("CLIP text representation configuration defined.")
display(clip_text_config_df)



### 🔷 Step 3 — Verify Runtime Environment and Dependencies

* Verify the active Python runtime, operating system, and PyTorch installation.
* Confirm that CUDA is available and validate the required NVIDIA L4 GPU when enabled.
* Display detected GPU hardware and available GPU memory.
* Verify that the Hugging Face Transformers library is installed and available.
* Confirm that the required CLIP model and processor classes can be imported successfully.
* Validate that the runtime environment satisfies all software and hardware requirements.
* Display a runtime verification summary before loading the CLIP text model.

In [ ]:
# ============================================================
# Step 3: Verify Runtime Environment and Dependencies
# ============================================================

import platform
import sys

import torch

print("Verifying runtime environment...")
print("-" * 60)

# ------------------------------------------------------------
# Python
# ------------------------------------------------------------

print(f"Python Version : {sys.version.split()[0]}")
print(f"Platform       : {platform.platform()}")

# ------------------------------------------------------------
# PyTorch
# ------------------------------------------------------------

print(f"PyTorch Version: {torch.__version__}")

# ------------------------------------------------------------
# CUDA
# ------------------------------------------------------------

cuda_available = torch.cuda.is_available()

print(f"CUDA Available : {cuda_available}")

if REQUIRE_L4_GPU:

    if not cuda_available:
        raise RuntimeError(
            "CUDA GPU is required for CLIP text representation generation."
        )

    gpu_name = torch.cuda.get_device_name(0)

    print(f"GPU            : {gpu_name}")

    if "L4" not in gpu_name:
        raise RuntimeError(
            f"NVIDIA L4 GPU required. Detected: {gpu_name}"
        )

    gpu_properties = torch.cuda.get_device_properties(0)

    gpu_memory_gb = (
        gpu_properties.total_memory
        / (1024 ** 3)
    )

    print(f"GPU Memory     : {gpu_memory_gb:.1f} GB")

# ------------------------------------------------------------
# Transformers
# ------------------------------------------------------------

try:
    import transformers

    print(f"Transformers   : {transformers.__version__}")

except ImportError:

    raise ImportError(
        "The transformers package is required."
    )

# ------------------------------------------------------------
# Verify CLIP Classes
# ------------------------------------------------------------

try:
    from transformers import (
        CLIPModel,
        CLIPProcessor,
    )

    print("CLIP classes   : Available")

except ImportError:

    raise ImportError(
        "Unable to import CLIPModel and CLIPProcessor."
    )

# ------------------------------------------------------------
# Runtime Summary
# ------------------------------------------------------------

print("\nRuntime verification complete.")
print("-" * 60)
print("Environment is ready for CLIP text representation generation.")



### 🔷 Step 4 — Prepare CLIP Text Input Dataset

* Select the configured development evaluation subset from the NExT-QA annotation dataset.
* Validate the required question, answer, and answer-choice fields for CLIP text representation generation.
* Generate a reproducible development subset using the configured evaluation split and random seed.
* Associate each evaluation question with its ground-truth answer text.
* Construct normalized text input records for both questions and multiple-choice answer options.
* Attach experiment metadata required for downstream representation processing.
* Validate the generated text input dataset prior to CLIP text embedding generation.
* Display summary statistics and representative text input records for verification.

In [ ]:
# ============================================================
# Step 4: Prepare CLIP Text Input Dataset
# ============================================================

import pandas as pd

print("Preparing CLIP text input dataset...")

# ------------------------------------------------------------
# Validate annotation columns
# ------------------------------------------------------------

required_annotation_columns = [
    "split",
    VIDEO_ID_COLUMN,
    QUESTION_COLUMN,
    GROUND_TRUTH_ANSWER_COLUMN,
    *CHOICE_COLUMNS,
]

missing_annotation_columns = [
    col for col in required_annotation_columns
    if col not in annotations_df.columns
]

if missing_annotation_columns:
    raise ValueError(
        f"annotations_df is missing required columns: {missing_annotation_columns}"
    )

# ------------------------------------------------------------
# Select development evaluation subset
# ------------------------------------------------------------

eval_df = annotations_df[
    annotations_df["split"] == evaluation_split
].copy()

if len(eval_df) == 0:
    raise ValueError(f"No records found for split: {evaluation_split}")

sample_size = min(
    development_subset_size,
    len(eval_df),
)

eval_df = (
    eval_df
    .sample(
        n=sample_size,
        random_state=random_seed,
    )
    .reset_index(drop=True)
)

eval_df[VIDEO_ID_COLUMN] = eval_df[VIDEO_ID_COLUMN].astype(str)

print(f"Selected evaluation records: {len(eval_df):,}")

# ------------------------------------------------------------
# Attach ground-truth answer text
# ------------------------------------------------------------

def answer_index_to_text(row):
    answer_idx = int(row[GROUND_TRUTH_ANSWER_COLUMN])
    option_col = f"a{answer_idx}"

    if option_col not in CHOICE_COLUMNS:
        raise ValueError(
            f"Answer option column not valid: {option_col}"
        )

    return row[option_col]

eval_df["ground_truth_text"] = eval_df.apply(
    answer_index_to_text,
    axis=1,
)

# ------------------------------------------------------------
# Build normalized text input records
# ------------------------------------------------------------

text_records = []

for row_index, row in eval_df.iterrows():

    question_id = row.get("question_id", row.get("qid", row_index))
    video_id = str(row[VIDEO_ID_COLUMN])

    # Question text record
    text_records.append(
        {
            "record_id": f"{video_id}_{question_id}_question",
            "video": video_id,
            "question_id": question_id,
            "text_type": "question",
            "choice_index": None,
            "text": row[QUESTION_COLUMN],
            "answer": int(row[GROUND_TRUTH_ANSWER_COLUMN]),
            "ground_truth_text": row["ground_truth_text"],
            "split": row["split"],
            "representation_experiment": EXPERIMENT_NAME,
            "representation_source": "clip_text",
        }
    )

    # Answer-choice text records
    for choice_index, choice_col in enumerate(CHOICE_COLUMNS):
        text_records.append(
            {
                "record_id": f"{video_id}_{question_id}_choice_{choice_index}",
                "video": video_id,
                "question_id": question_id,
                "text_type": "answer_choice",
                "choice_index": choice_index,
                "text": row[choice_col],
                "answer": int(row[GROUND_TRUTH_ANSWER_COLUMN]),
                "ground_truth_text": row["ground_truth_text"],
                "split": row["split"],
                "representation_experiment": EXPERIMENT_NAME,
                "representation_source": "clip_text",
            }
        )

text_input_df = pd.DataFrame(text_records)

if text_input_df.empty:
    raise RuntimeError("No CLIP text input records were generated.")

# ------------------------------------------------------------
# Validate generated text inputs
# ------------------------------------------------------------

required_text_columns = [
    "record_id",
    "video",
    "question_id",
    "text_type",
    "choice_index",
    "text",
    "answer",
    "ground_truth_text",
    "split",
    "representation_experiment",
    "representation_source",
]

missing_text_columns = [
    col for col in required_text_columns
    if col not in text_input_df.columns
]

if missing_text_columns:
    raise ValueError(
        f"text_input_df is missing required columns: {missing_text_columns}"
    )

empty_text_count = (
    text_input_df["text"]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

if empty_text_count > 0:
    raise ValueError(
        f"Found {empty_text_count} empty text input records."
    )

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

text_type_summary_df = (
    text_input_df
    .groupby("text_type")
    .size()
    .reset_index(name="record_count")
)

print("\nCLIP text input dataset prepared successfully.")
print(f"Evaluation records : {len(eval_df):,}")
print(f"Text input records : {len(text_input_df):,}")
print(f"Unique videos      : {eval_df[VIDEO_ID_COLUMN].nunique():,}")
print(f"Answer mode        : {ANSWER_MODE}")

print("\nText Type Summary:")
display(text_type_summary_df)

print("\nText Input Preview (First Question):")

first_question_id = text_input_df["question_id"].iloc[0]

display(
    text_input_df[
        text_input_df["question_id"] == first_question_id
    ]
)



### 🔷 Step 5 — Load CLIP Text Model

* Select the appropriate computation device for CLIP text representation generation.
* Load the pretrained CLIP text encoder model from the Hugging Face Transformers library.
* Load the corresponding CLIP processor used for text tokenization and preprocessing.
* Configure the CLIP model for inference by switching to evaluation mode.
* Verify the text embedding dimension and maximum supported text sequence length.
* Perform a sample inference to confirm successful text embedding generation.
* Display model configuration details and verification results prior to batch representation generation.

In [ ]:
# ============================================================
# Step 5: Load CLIP Text Model
# ============================================================

import torch

from transformers import (
    CLIPModel,
    CLIPProcessor,
)

print("Loading CLIP text model...")

# ------------------------------------------------------------
# Select computation device
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"Device: {device}")

# ------------------------------------------------------------
# Load CLIP model
# ------------------------------------------------------------

clip_model = CLIPModel.from_pretrained(
    CLIP_TEXT_MODEL_NAME
)

clip_model.to(device)
clip_model.eval()

# ------------------------------------------------------------
# Load CLIP processor
# ------------------------------------------------------------

clip_processor = CLIPProcessor.from_pretrained(
    CLIP_TEXT_MODEL_NAME
)

# ------------------------------------------------------------
# Verify model configuration
# ------------------------------------------------------------

text_embedding_dimension = (
    clip_model.config.projection_dim
)

text_max_position_embeddings = (
    clip_model.text_model.config.max_position_embeddings
)

print("\nCLIP text model loaded successfully.")
print(f"Model                     : {CLIP_TEXT_MODEL_NAME}")
print(f"Embedding dimension       : {text_embedding_dimension}")
print(f"Maximum text length       : {text_max_position_embeddings}")
print(f"Model device              : {device}")

# ------------------------------------------------------------
# Verify inference
# ------------------------------------------------------------

sample_inputs = clip_processor(
    text=["CLIP model verification"],
    return_tensors="pt",
    padding=True,
)

sample_inputs = {
    key: value.to(device)
    for key, value in sample_inputs.items()
}

with torch.no_grad():
    sample_outputs = clip_model.text_model(
        **sample_inputs
    )

    sample_features = sample_outputs.pooler_output

print(
    f"Verification embedding shape : "
    f"{tuple(sample_features.shape)}"
)

print("\nCLIP text model is ready for representation generation.")



### 🔷 Step 6 — Generate CLIP Text Representations

* Generate CLIP text embeddings for each prepared question and multiple-choice answer option.
* Process text inputs in batches to improve inference performance and GPU utilization.
* Tokenize and encode text using the pretrained CLIP text encoder.
* Normalize embedding vectors to produce consistent representation magnitudes.
* Associate each embedding with its corresponding text record and experiment metadata.
* Construct the complete CLIP text representation dataset for downstream VideoQA experiments.
* Verify that the expected embedding dimensions were generated successfully.
* Display summary statistics describing the completed text representation generation process.


In [ ]:
# ============================================================
# Step 6: Generate CLIP Text Representations
# ============================================================

import time
import numpy as np
import pandas as pd
import torch
from tqdm.notebook import tqdm

print("Generating CLIP text embeddings...")

if "text_input_df" not in globals():
    raise NameError("text_input_df was not found. Run Step 4 first.")

if "clip_model" not in globals():
    raise NameError("clip_model was not found. Run Step 5 first.")

if "clip_processor" not in globals():
    raise NameError("clip_processor was not found. Run Step 5 first.")

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------

CLIP_TEXT_BATCH_SIZE = 32

clip_model.eval()

records = []
start_time = time.time()

# ------------------------------------------------------------
# Batch text encoding
# ------------------------------------------------------------

for start_idx in tqdm(
    range(0, len(text_input_df), CLIP_TEXT_BATCH_SIZE),
    desc="Encoding CLIP text",
):
    batch_df = text_input_df.iloc[
        start_idx:start_idx + CLIP_TEXT_BATCH_SIZE
    ].copy()

    batch_texts = (
        batch_df["text"]
        .astype(str)
        .tolist()
    )

    inputs = clip_processor(
        text=batch_texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        text_outputs = clip_model.text_model(
            **inputs
        )

        pooled_output = text_outputs.pooler_output

        text_features = clip_model.text_projection(
            pooled_output
        )

        text_features = text_features / text_features.norm(
            dim=-1,
            keepdim=True,
        )

    text_features_np = (
        text_features
        .detach()
        .cpu()
        .numpy()
    )

    for row, embedding in zip(
        batch_df.to_dict("records"),
        text_features_np,
    ):
        record = dict(row)

        for i, value in enumerate(embedding):
            record[f"clip_text_{i:03d}"] = float(value)

        records.append(record)

elapsed_time = time.time() - start_time

clip_text_representation_df = pd.DataFrame(records)

if clip_text_representation_df.empty:
    raise RuntimeError("No CLIP text representations were generated.")

# ------------------------------------------------------------
# Identify embedding columns
# ------------------------------------------------------------

clip_text_columns = [
    col for col in clip_text_representation_df.columns
    if col.startswith("clip_text_")
]

if len(clip_text_columns) == 0:
    raise ValueError("No CLIP text embedding columns were generated.")

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

print("\nCLIP text embedding generation complete.")
print(f"Input text records   : {len(text_input_df):,}")
print(f"Embedding records    : {len(clip_text_representation_df):,}")
print(f"Embedding dimensions : {len(clip_text_columns):,}")
print(f"Batch size           : {CLIP_TEXT_BATCH_SIZE}")
print(f"Elapsed time         : {elapsed_time:.1f} seconds")
print(
    f"Average/record       : "
    f"{elapsed_time / len(clip_text_representation_df):.3f} seconds"
)



### 🔷 Step 7 — Validate CLIP Text Representation Dataset

* Verify that the CLIP text representation dataset was generated successfully.
* Validate the number of question records, answer-choice records, videos, and unique question records.
* Confirm that the expected CLIP text embedding dimensions were produced.
* Verify that all embedding values are present and numeric.
* Check for duplicate representation records within the generated dataset.
* Confirm that experiment metadata was propagated correctly to all representation records.
* Display a validation summary describing the integrity of the generated CLIP text representation dataset.
* Verify that the representation dataset is ready for downstream VideoQA experiments.

In [ ]:
# ============================================================
# Step 7: Validate CLIP Text Representation Dataset
# ============================================================

import pandas as pd

print("Validating CLIP text representation dataset...")

if "clip_text_representation_df" not in globals():
    raise NameError(
        "clip_text_representation_df was not found. Run Step 6 first."
    )

if "clip_text_columns" not in globals():
    raise NameError(
        "clip_text_columns was not found. Run Step 6 first."
    )

# ------------------------------------------------------------
# Validation Summary
# ------------------------------------------------------------

validation_summary = {
    "representation_records": len(
        clip_text_representation_df
    ),
    "question_records": (
        clip_text_representation_df["text_type"]
        .eq("question")
        .sum()
    ),
    "answer_choice_records": (
        clip_text_representation_df["text_type"]
        .eq("answer_choice")
        .sum()
    ),
    "unique_videos": (
        clip_text_representation_df["video"]
        .nunique()
    ),
    "unique_question_records": (
        clip_text_representation_df
        .loc[
            clip_text_representation_df["text_type"] == "question",
            ["video", "question_id"]
        ]
        .drop_duplicates()
        .shape[0]
    ),
    "embedding_dimensions": len(
        clip_text_columns
    ),
    "missing_embedding_values": (
        clip_text_representation_df[
            clip_text_columns
        ]
        .isna()
        .sum()
        .sum()
    ),
    "non_numeric_embedding_columns": sum(
        not pd.api.types.is_numeric_dtype(
            clip_text_representation_df[col]
        )
        for col in clip_text_columns
    ),
    "duplicate_record_ids": (
        clip_text_representation_df["record_id"]
        .duplicated()
        .sum()
    ),
    "representation_experiment": (
        clip_text_representation_df[
            "representation_experiment"
        ]
        .iloc[0]
    ),
}

validation_df = pd.DataFrame(
    validation_summary.items(),
    columns=["Validation Check", "Value"],
)

display(validation_df)

# ------------------------------------------------------------
# Validation Checks
# ------------------------------------------------------------

if validation_summary["missing_embedding_values"] > 0:
    raise ValueError(
        "Missing values detected in CLIP text representations."
    )

if validation_summary["non_numeric_embedding_columns"] > 0:
    raise ValueError(
        "Non-numeric embedding columns detected."
    )

if validation_summary["duplicate_record_ids"] > 0:
    raise ValueError(
        "Duplicate record IDs detected."
    )

print(
    "\nRepresentation validation passed. "
    "No missing, non-numeric, or duplicate records detected."
)



### 🔷 Step 8 — Save CLIP Text Representation Files

* Create the experiment output directory for CLIP text representation artifacts when necessary.
* Save the generated CLIP text representation dataset to the experiment-specific Google Drive location.
* Verify that the representation dataset was written successfully.
* Confirm the number of generated representation records and embedding dimensions.
* Summarize the number of question and answer-choice representations saved.
* Display the output file location for downstream representation-based VideoQA experiments.


In [ ]:
# ============================================================
# Step 8: Save CLIP Text Representation Files
# ============================================================

print("Saving CLIP text representation files...")

if "clip_text_representation_df" not in globals():
    raise NameError(
        "clip_text_representation_df was not found. Run Step 6 first."
    )

if "clip_text_columns" not in globals():
    raise NameError(
        "clip_text_columns was not found. Run Step 6 first."
    )

# ------------------------------------------------------------
# Create output directory
# ------------------------------------------------------------

CLIP_TEXT_REPRESENTATIONS_DRIVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# ------------------------------------------------------------
# Save representation dataset
# ------------------------------------------------------------

clip_text_representation_df.to_csv(
    CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV,
    index=False,
)

# ------------------------------------------------------------
# Verify output
# ------------------------------------------------------------

if not CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV.exists():
    raise FileNotFoundError(
        f"Failed to create: {CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV}"
    )

# ------------------------------------------------------------
# Display Summary
# ------------------------------------------------------------

print("\nCLIP text representation dataset saved successfully.")
print(f"Output file          : {CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV}")
print(f"Representation rows  : {len(clip_text_representation_df):,}")
print(f"Embedding dimensions : {len(clip_text_columns):,}")
print(
    f"Question records     : "
    f"{(clip_text_representation_df['text_type'] == 'question').sum():,}"
)
print(
    f"Answer-choice records: "
    f"{(clip_text_representation_df['text_type'] == 'answer_choice').sum():,}"
)



### 🔷 Step 9 — Generate CLIP Text Representation Summary Report

* Generate summary statistics describing the completed CLIP text representation dataset.
* Summarize the number of generated question and answer-choice representations.
* Report the number of unique videos and unique question records included in the development subset.
* Verify the generated CLIP text embedding dimensionality and check for missing embedding values.
* Record experiment configuration metadata required for downstream representation-based VideoQA experiments.
* Save the representation summary report to the experiment-specific Google Drive output directory.
* Display the completed CLIP text representation summary for experiment verification.


In [ ]:
# ============================================================
# Step 9: Generate CLIP Text Representation Summary Report
# ============================================================

import pandas as pd

print("Generating CLIP text representation summary report...")

if "clip_text_representation_df" not in globals():
    raise NameError(
        "clip_text_representation_df was not found. Run Step 6 first."
    )

if "clip_text_columns" not in globals():
    raise NameError(
        "clip_text_columns was not found. Run Step 6 first."
    )

# ------------------------------------------------------------
# Compute summary statistics
# ------------------------------------------------------------

question_record_count = (
    clip_text_representation_df["text_type"]
    .eq("question")
    .sum()
)

answer_choice_record_count = (
    clip_text_representation_df["text_type"]
    .eq("answer_choice")
    .sum()
)

unique_question_records = (
    clip_text_representation_df
    .loc[
        clip_text_representation_df["text_type"] == "question",
        ["video", "question_id"]
    ]
    .drop_duplicates()
    .shape[0]
)

missing_embedding_values = (
    clip_text_representation_df[clip_text_columns]
    .isna()
    .sum()
    .sum()
)

summary_rows = [
    {"metric": "experiment_name", "value": EXPERIMENT_NAME},
    {"metric": "experiment_type", "value": "clip_text_representation"},
    {"metric": "clip_text_model", "value": CLIP_TEXT_MODEL_NAME},
    {"metric": "representation_scope", "value": CLIP_TEXT_REPRESENTATION_SCOPE},
    {"metric": "answer_mode", "value": ANSWER_MODE},
    {"metric": "representation_records", "value": len(clip_text_representation_df)},
    {"metric": "question_records", "value": int(question_record_count)},
    {"metric": "answer_choice_records", "value": int(answer_choice_record_count)},
    {"metric": "unique_videos", "value": clip_text_representation_df["video"].nunique()},
    {"metric": "unique_question_records", "value": int(unique_question_records)},
    {"metric": "embedding_dimensions", "value": len(clip_text_columns)},
    {"metric": "missing_embedding_values", "value": int(missing_embedding_values)},
]

clip_text_summary_df = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# Save summary report
# ------------------------------------------------------------

CLIP_TEXT_REPRESENTATIONS_DRIVE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

clip_text_summary_df.to_csv(
    CLIP_TEXT_SUMMARY_DRIVE_CSV,
    index=False,
)

if not CLIP_TEXT_SUMMARY_DRIVE_CSV.exists():
    raise FileNotFoundError(
        f"Failed to create: {CLIP_TEXT_SUMMARY_DRIVE_CSV}"
    )

# ------------------------------------------------------------
# Display summary
# ------------------------------------------------------------

print("CLIP text representation summary report saved.")
print(f"Summary file : {CLIP_TEXT_SUMMARY_DRIVE_CSV}")

display(clip_text_summary_df)



### 🔷 Step 10 — Display Sample CLIP Text Representation Records

* Randomly select representative CLIP text representation records from the generated development dataset.
* Display question and answer-choice representations together with their associated experiment metadata.
* Display a subset of embedding dimensions for representative inspection.
* Summarize the number of generated representation records, question records, and answer-choice records.
* Report the number of unique videos and unique question records included in the development subset.
* Confirm the dimensionality of the generated CLIP text representations.
* Provide a qualitative verification of the generated representation dataset prior to downstream VideoQA experiments.

In [ ]:
# ============================================================
# Step 10: Display Sample Text Representation Records
# ============================================================

import pandas as pd

print("Displaying sample CLIP text representation records...")

if "clip_text_representation_df" not in globals():
    raise NameError(
        "clip_text_representation_df was not found. Run Step 6 first."
    )

if "clip_text_columns" not in globals():
    raise NameError(
        "clip_text_columns was not found. Run Step 6 first."
    )

# ------------------------------------------------------------
# Select sample records
# ------------------------------------------------------------

sample_count = min(
    10,
    len(clip_text_representation_df),
)

sample_text_representation_df = (
    clip_text_representation_df
    .sample(
        n=sample_count,
        random_state=RANDOM_SEED,
    )
    .reset_index(drop=True)
)

display_columns = [
    "record_id",
    "video",
    "question_id",
    "text_type",
    "choice_index",
    "text",
    "answer",
    "ground_truth_text",
    "representation_source",
    "representation_experiment",
    *clip_text_columns[:5],
]

print(f"Displaying {sample_count} CLIP text representation records...")
print(f"CLIP model            : {CLIP_TEXT_MODEL_NAME}")
print(f"Representation source : clip_text")
print(f"Embedding dimensions  : {len(clip_text_columns)}")
print("\nShowing first 5 embedding dimensions only.\n")

display(
    sample_text_representation_df[
        display_columns
    ]
)

# ------------------------------------------------------------
# Representation Summary
# ------------------------------------------------------------

question_record_count = (
    clip_text_representation_df["text_type"]
    .eq("question")
    .sum()
)

answer_choice_record_count = (
    clip_text_representation_df["text_type"]
    .eq("answer_choice")
    .sum()
)

unique_question_records = (
    clip_text_representation_df
    .loc[
        clip_text_representation_df["text_type"] == "question",
        ["video", "question_id"]
    ]
    .drop_duplicates()
    .shape[0]
)

print("\nCLIP Text Representation Summary")
print("-" * 60)
print(f"Representation records   : {len(clip_text_representation_df):,}")
print(f"Question records         : {question_record_count:,}")
print(f"Answer-choice records    : {answer_choice_record_count:,}")
print(f"Unique videos            : {clip_text_representation_df['video'].nunique():,}")
print(f"Unique question records  : {unique_question_records:,}")
print(f"Embedding dimensions     : {len(clip_text_columns):,}")



### 🔷 Step 11 — Notebook Summary

* Summarize the completed CLIP text representation generation experiment.
* Report the active experiment configuration, CLIP text model, evaluation split, development subset size, and answer mode.
* Summarize the generated question and answer-choice representation records.
* Report the number of unique videos, unique question records, and generated embedding dimensions.
* List the CLIP text representation artifacts generated by the notebook and their experiment-specific output locations.
* Confirm that the generated CLIP text representations are ready for downstream representation-based VideoQA experiments.

In [ ]:
# ============================================================
# Step 11: Notebook Summary
# ============================================================

print("Notebook 05 complete.")
print("=" * 60)

print("\nCLIP Text Representation Experiment")
print("-" * 60)
print(f"Experiment name          : {EXPERIMENT_NAME}")
print(f"CLIP text model          : {CLIP_TEXT_MODEL_NAME}")
print(f"Evaluation split         : {EVALUATION_SPLIT}")
print(f"Development subset size  : {DEVELOPMENT_SUBSET_SIZE}")
print(f"Answer mode              : {ANSWER_MODE}")

print("\nText Representation Dataset")
print("-" * 60)
print(f"Representation records   : {len(clip_text_representation_df):,}")
print(
    f"Question records         : "
    f"{(clip_text_representation_df['text_type'] == 'question').sum():,}"
)
print(
    f"Answer-choice records    : "
    f"{(clip_text_representation_df['text_type'] == 'answer_choice').sum():,}"
)
print(f"Unique videos            : {clip_text_representation_df['video'].nunique():,}")

unique_question_records = (
    clip_text_representation_df
    .loc[
        clip_text_representation_df["text_type"] == "question",
        ["video", "question_id"]
    ]
    .drop_duplicates()
    .shape[0]
)

print(f"Unique question records  : {unique_question_records:,}")
print(f"Embedding dimensions     : {len(clip_text_columns):,}")

print("\nGenerated Outputs")
print("-" * 60)
print(f"Text representations     : {CLIP_TEXT_REPRESENTATIONS_DRIVE_CSV}")
print(f"Representation summary   : {CLIP_TEXT_SUMMARY_DRIVE_CSV}")
print(f"Output directory         : {CLIP_TEXT_REPRESENTATIONS_DRIVE_DIR}")

print("\nNotebook 05 generated:")
print("- CLIP text representation dataset")
print("- CLIP text representation summary")
print("- Validated text representation records")
print("- Sample text representation records")

print("\nNotebook 05 is ready for downstream")
print("representation-based VideoQA classification.")

